# EconLens: UK Economic Time-Series Forecasting

EconLens compares benchmark and statistical forecasting models across the same eight Bank of England series used in the UK Economic Data Pipeline project.

Its focus is chronological holdout comparison, transparent model selection, residual diagnostics and interpretable forecasts. The optional natural-language interface is deterministic keyword routing—not an LLM or a deployed chatbot.

## What this notebook demonstrates

1. Native-frequency time-series preparation
2. Chronological training/holdout splits
3. Training-only ADF stationarity checks
4. Naïve, seasonal-naïve and drift baselines
5. Holt, seasonal ETS and SARIMA candidates
6. MAE, RMSE and sMAPE comparison
7. Full-series refitting and forecast generation
8. Model and approximate residual-based prediction intervals
9. Ljung–Box and Jarque–Bera residual diagnostics
10. Transparent keyword-based query routing

Reported errors come from one model-selection holdout, not an untouched final test set or rolling-origin evaluation.


## Dataset catalogue

| Series code | EconLens label | Frequency used | Forecast horizon |
|---|---|---:|---:|
| LPMAUYN | M4 / monetary aggregate | Monthly | 12 months |
| IUDBEDR | Official Bank Rate | Business daily | 30 business days |
| IUDSOIA | SONIA | Business daily | 30 business days |
| IUMBV34 | 2-year fixed mortgage rate | Monthly | 12 months |
| IUMBV42 | 5-year fixed mortgage rate | Monthly | 12 months |
| LPMVZRI | Consumer credit | Monthly | 12 months |
| XUDLUSS | GBP/USD | Business daily | 30 business days |
| XUDLERS | GBP/EUR | Business daily | 30 business days |

The filenames match the cleaned UK Economic Data Pipeline notebook. Place the eight CSV files in `data/raw/` and verify their official Bank of England metadata before publication.


## 1. Environment setup


In [1]:
import re
import warnings
import sys
from pathlib import Path
from typing import Any

import numpy as np
from IPython.display import display
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')


Python: 3.12.13
pandas: 3.0.5


## 2. Project paths


In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'Dataset folder not found: {DATA_DIR}. '
        'See data/README.md for the required Bank of England files.'
    )

print(f'Input datasets: {DATA_DIR}')
print(f'EconLens outputs: {OUTPUT_DIR}')

IMAGES_DIR = PROJECT_ROOT / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


Input datasets: C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\data\raw
EconLens outputs: C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs


## 3. Dataset and modelling configuration


In [3]:
DATASET_CONFIG = {
    'LPMAUYN': {
        'filename': 'LPMAUYN  Bank of England  Database.csv',
        'value_column': 'LPMAUYN Amount',
        'label': 'M4 / monetary aggregate',
        'category': 'Money and credit',
        'frequency': 'MS',
        'frequency_label': 'monthly',
        'seasonal_period': 12,
        'test_size': 12,
        'forecast_horizon': 12,
        'transform': 'log',
    },
    'IUDBEDR': {
        'filename': 'IUDBEDR  Bank of England  Database.csv',
        'value_column': 'IUDBEDR Rate',
        'label': 'Official Bank Rate',
        'category': 'Interest rates',
        'frequency': 'B',
        'frequency_label': 'business daily',
        'seasonal_period': 5,
        'test_size': 30,
        'forecast_horizon': 30,
        'transform': 'none',
    },
    'IUDSOIA': {
        'filename': 'IUDSOIA  Bank of England  Database.csv',
        'value_column': 'IUDSOIA Rate',
        'label': 'SONIA',
        'category': 'Interest rates',
        'frequency': 'B',
        'frequency_label': 'business daily',
        'seasonal_period': 5,
        'test_size': 30,
        'forecast_horizon': 30,
        'transform': 'none',
    },
    'IUMBV34': {
        'filename': 'IUMBV34  Bank of England  Database.csv',
        'value_column': 'IUMBV34 Rate',
        'label': '2-year fixed mortgage rate',
        'category': 'Mortgage market',
        'frequency': 'MS',
        'frequency_label': 'monthly',
        'seasonal_period': 12,
        'test_size': 12,
        'forecast_horizon': 12,
        'transform': 'none',
    },
    'IUMBV42': {
        'filename': 'IUMBV42  Bank of England  Database.csv',
        'value_column': 'IUMBV42 Rate',
        'label': '5-year fixed mortgage rate',
        'category': 'Mortgage market',
        'frequency': 'MS',
        'frequency_label': 'monthly',
        'seasonal_period': 12,
        'test_size': 12,
        'forecast_horizon': 12,
        'transform': 'none',
    },
    'LPMVZRI': {
        'filename': 'LPMVZRI  Bank of England  Database.csv',
        'value_column': 'LPMVZRI Amount',
        'label': 'Consumer credit',
        'category': 'Money and credit',
        'frequency': 'MS',
        'frequency_label': 'monthly',
        'seasonal_period': 12,
        'test_size': 12,
        'forecast_horizon': 12,
        'transform': 'log',
    },
    'XUDLUSS': {
        'filename': 'XUDLUSS  Bank of England  Database.csv',
        'value_column': 'XUDLUSS Rate',
        'label': 'GBP/USD',
        'category': 'Exchange rates',
        'frequency': 'B',
        'frequency_label': 'business daily',
        'seasonal_period': 5,
        'test_size': 30,
        'forecast_horizon': 30,
        'transform': 'none',
    },
    'XUDLERS': {
        'filename': 'XUDLERS  Bank of England  Database.csv',
        'value_column': 'XUDLERS Rate',
        'label': 'GBP/EUR',
        'category': 'Exchange rates',
        'frequency': 'B',
        'frequency_label': 'business daily',
        'seasonal_period': 5,
        'test_size': 30,
        'forecast_horizon': 30,
        'transform': 'none',
    },
}

config_table = pd.DataFrame(DATASET_CONFIG).T[
    ['label', 'category', 'frequency_label', 'seasonal_period', 'test_size', 'forecast_horizon', 'transform']
]
display(config_table)


,label,category,frequency_label,seasonal_period,test_size,forecast_horizon,transform
LPMAUYN,M4 / monetary aggregate,Money and credit,monthly,12,12,12,log
IUDBEDR,Official Bank Rate,Interest rates,business daily,5,30,30,none
IUDSOIA,SONIA,Interest rates,business daily,5,30,30,none
IUMBV34,2-year fixed mortgage rate,Mortgage market,monthly,12,12,12,none
IUMBV42,5-year fixed mortgage rate,Mortgage market,monthly,12,12,12,none
LPMVZRI,Consumer credit,Money and credit,monthly,12,12,12,log
XUDLUSS,GBP/USD,Exchange rates,business daily,5,30,30,none
XUDLERS,GBP/EUR,Exchange rates,business daily,5,30,30,none


## 4. Data ingestion and standardisation


In [4]:
def load_and_standardise_series(series_code: str, config: dict[str, Any]) -> pd.DataFrame:
    """Load one Bank of England CSV and return a clean Date/value DataFrame."""
    file_path = DATA_DIR / config['filename']
    if not file_path.exists():
        raise FileNotFoundError(f'{series_code}: missing file {file_path}')

    frame = pd.read_csv(file_path)
    if 'Date' not in frame.columns:
        raise ValueError(f"{series_code}: expected a 'Date' column, found {list(frame.columns)}")

    value_candidates = [column for column in frame.columns if column != 'Date']
    if len(value_candidates) != 1:
        raise ValueError(
            f'{series_code}: expected exactly one non-Date column, found {value_candidates}'
        )

    clean = frame.rename(columns={value_candidates[0]: config['value_column']}).copy()
    clean['Date'] = pd.to_datetime(clean['Date'], format='mixed', dayfirst=True, errors='coerce')
    clean[config['value_column']] = pd.to_numeric(
        clean[config['value_column']], errors='coerce'
    )

    clean = (
        clean.dropna(subset=['Date', config['value_column']])
        .drop_duplicates(subset='Date', keep='last')
        .sort_values('Date')
        .reset_index(drop=True)
    )

    if clean.empty:
        raise ValueError(f'{series_code}: no usable observations after cleaning.')
    return clean


raw_series = {
    code: load_and_standardise_series(code, config)
    for code, config in DATASET_CONFIG.items()
}

load_summary = pd.DataFrame([
    {
        'series_code': code,
        'label': DATASET_CONFIG[code]['label'],
        'rows': len(frame),
        'start_date': frame['Date'].min(),
        'end_date': frame['Date'].max(),
        'missing_values': int(frame.isna().sum().sum()),
        'duplicate_dates': int(frame['Date'].duplicated().sum()),
    }
    for code, frame in raw_series.items()
])

display(load_summary)


,series_code,label,rows,start_date,end_date,missing_values,duplicate_dates
0,LPMAUYN,M4 / monetary aggregate,65,2021-01-31,2026-05-31,0,0
1,IUDBEDR,Official Bank Rate,1402,2021-01-04,2026-07-23,0,0
2,IUDSOIA,SONIA,1401,2021-01-04,2026-07-22,0,0
3,IUMBV34,2-year fixed mortgage rate,66,2021-01-31,2026-06-30,0,0
4,IUMBV42,5-year fixed mortgage rate,66,2021-01-31,2026-06-30,0,0
5,LPMVZRI,Consumer credit,51,2021-01-31,2025-03-31,0,0
6,XUDLUSS,GBP/USD,1402,2021-01-04,2026-07-23,0,0
7,XUDLERS,GBP/EUR,1402,2021-01-04,2026-07-23,0,0


In [5]:
def prepare_model_series(
    frame: pd.DataFrame,
    value_column: str,
    frequency: str,
) -> pd.Series:
    """
    Align a series to its modelling frequency.

    - Monthly series use the final available observation in each month.
    - Business-daily series use the final available observation on each business day.
    - Forward filling handles publication gaps and market holidays without using future data.
    """
    series = (
        frame.set_index('Date')[value_column]
        .astype(float)
        .sort_index()
        .resample(frequency)
        .last()
        .ffill()
        .dropna()
    )
    series.name = value_column

    if not series.index.is_monotonic_increasing:
        raise ValueError(f'{value_column}: index is not chronological.')
    if series.index.has_duplicates:
        raise ValueError(f'{value_column}: duplicate dates remain after resampling.')
    if len(series) < 36:
        raise ValueError(f'{value_column}: only {len(series)} observations; at least 36 are required.')
    return series


model_series = {
    code: prepare_model_series(
        raw_series[code],
        config['value_column'],
        config['frequency'],
    )
    for code, config in DATASET_CONFIG.items()
}

prepared_summary = pd.DataFrame([
    {
        'series_code': code,
        'label': DATASET_CONFIG[code]['label'],
        'frequency': DATASET_CONFIG[code]['frequency_label'],
        'observations': len(series),
        'start_date': series.index.min(),
        'end_date': series.index.max(),
        'minimum': series.min(),
        'maximum': series.max(),
    }
    for code, series in model_series.items()
])

display(prepared_summary)


,series_code,label,frequency,observations,start_date,end_date,minimum,maximum
0,LPMAUYN,M4 / monetary aggregate,monthly,65,2021-01-01,2026-05-01,"2,832,564.0000","3,278,498.0000"
1,IUDBEDR,Official Bank Rate,business daily,1449,2021-01-04,2026-07-23,0.1000,5.2500
2,IUDSOIA,SONIA,business daily,1448,2021-01-04,2026-07-22,0.0450,5.2001
3,IUMBV34,2-year fixed mortgage rate,monthly,66,2021-01-01,2026-06-01,1.2000,6.2200
4,IUMBV42,5-year fixed mortgage rate,monthly,66,2021-01-01,2026-06-01,1.2800,5.7100
5,LPMVZRI,Consumer credit,monthly,51,2021-01-01,2025-03-01,"375,935.0000","530,094.0000"
6,XUDLUSS,GBP/USD,business daily,1449,2021-01-04,2026-07-23,1.0745,1.4211
7,XUDLERS,GBP/EUR,business daily,1449,2021-01-04,2026-07-23,1.1032,1.2148


## 5. Initial time-series exploration


In [6]:
def plot_series(series_code: str) -> go.Figure:
    config = DATASET_CONFIG[series_code]
    series = model_series[series_code]
    figure = go.Figure()
    figure.add_trace(go.Scatter(x=series.index, y=series.values, mode='lines', name=config['label']))
    figure.update_layout(
        title=f"{config['label']} — modelling series",
        xaxis_title='Date',
        yaxis_title=config['value_column'],
        hovermode='x unified',
        template='plotly_white',
    )
    return figure


for series_code in DATASET_CONFIG:
    plot_series(series_code).show()


## 6. Transformations and stationarity

The two amount series use a logarithmic transformation because their levels are strictly positive and much larger than the rate series. Model selection and evaluation are still reported on the original scale.

The Augmented Dickey-Fuller test is applied only to each training partition. EconLens chooses the first differencing order from 0 to 2 whose p-value is below 0.05. This keeps the holdout set completely unseen during model configuration.


In [7]:
def to_model_scale(series: pd.Series, transform: str) -> pd.Series:
    series = series.astype(float)
    if transform == 'log':
        if (series <= 0).any():
            raise ValueError(f'{series.name}: log transformation requires positive values.')
        return np.log(series)
    if transform == 'none':
        return series.copy()
    raise ValueError(f'Unsupported transform: {transform}')


def from_model_scale(values: pd.Series | np.ndarray, transform: str):
    if transform == 'log':
        return np.exp(values)
    if transform == 'none':
        return values
    raise ValueError(f'Unsupported transform: {transform}')


def choose_differencing_order(
    training_series: pd.Series,
    alpha: float = 0.05,
    maximum_d: int = 2,
) -> tuple[int, pd.DataFrame]:
    """Select d using sequential ADF tests on the training data only."""
    current = training_series.dropna().astype(float)
    rows = []

    for d in range(maximum_d + 1):
        if current.nunique() <= 1:
            rows.append({'d': d, 'adf_statistic': np.nan, 'p_value': 0.0, 'observations': len(current)})
            return d, pd.DataFrame(rows)

        try:
            result = adfuller(current, autolag='AIC', result_object=False)
            statistic, p_value, _, observations = result[:4]
        except Exception:
            statistic, p_value, observations = np.nan, 1.0, len(current)

        rows.append({
            'd': d,
            'adf_statistic': statistic,
            'p_value': p_value,
            'observations': observations,
        })
        if p_value < alpha:
            return d, pd.DataFrame(rows)
        current = current.diff().dropna()

    return maximum_d, pd.DataFrame(rows)


## 7. Forecast evaluation utilities


In [8]:
def smape(actual: pd.Series, predicted: pd.Series) -> float:
    actual_values = np.asarray(actual, dtype=float)
    predicted_values = np.asarray(predicted, dtype=float)
    denominator = np.abs(actual_values) + np.abs(predicted_values)
    valid = denominator > 1e-12
    if not valid.any():
        return 0.0
    return float(np.mean(200.0 * np.abs(predicted_values[valid] - actual_values[valid]) / denominator[valid]))


def forecast_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    actual_values = np.asarray(actual, dtype=float)
    predicted_values = np.asarray(predicted, dtype=float)
    errors = actual_values - predicted_values
    return {
        'MAE': float(np.mean(np.abs(errors))),
        'RMSE': float(np.sqrt(np.mean(errors ** 2))),
        'sMAPE_pct': smape(actual, predicted),
    }


def drift_forecast(training_series: pd.Series, steps: int) -> np.ndarray:
    if len(training_series) < 2:
        return np.repeat(training_series.iloc[-1], steps)
    slope = (training_series.iloc[-1] - training_series.iloc[0]) / (len(training_series) - 1)
    return training_series.iloc[-1] + slope * np.arange(1, steps + 1)


def seasonal_naive_forecast(training_series: pd.Series, steps: int, period: int) -> np.ndarray:
    pattern = training_series.iloc[-period:].to_numpy()
    return np.resize(pattern, steps)


def build_sarima_candidates(d: int, seasonal_period: int, allow_seasonality: bool):
    candidates = [
        ((0, d, 0), (0, 0, 0, 0)),
        ((1, d, 0), (0, 0, 0, 0)),
        ((0, d, 1), (0, 0, 0, 0)),
        ((1, d, 1), (0, 0, 0, 0)),
        ((2, d, 0), (0, 0, 0, 0)),
        ((0, d, 2), (0, 0, 0, 0)),
    ]
    if allow_seasonality:
        candidates.extend([
            ((1, d, 0), (1, 0, 0, seasonal_period)),
            ((0, d, 1), (0, 0, 1, seasonal_period)),
            ((1, d, 1), (0, 1, 1, seasonal_period)),
        ])
    return candidates


## 8. Candidate-model comparison

Every series is compared against simple benchmarks because a complex model should earn its place through better holdout performance.

- **Naïve:** repeats the latest observation.
- **Seasonal naïve:** repeats the latest seasonal pattern.
- **Drift:** extends the average change across the training sample.
- **Holt / ETS:** estimates level, trend and optional seasonality.
- **SARIMA:** evaluates a deliberately small, interpretable candidate grid.

Models are ranked on one chronological model-selection holdout using RMSE, then MAE and sMAPE. Because the same holdout selects and reports the winner, these metrics are comparison results rather than an unbiased final estimate of future performance.


In [9]:
def evaluate_model_candidates(
    training_original: pd.Series,
    test_original: pd.Series,
    config: dict[str, Any],
    differencing_order: int,
) -> tuple[pd.DataFrame, dict[str, pd.Series]]:
    transform = config['transform']
    training = to_model_scale(training_original, transform)
    horizon = len(test_original)
    period = int(config['seasonal_period'])

    rows: list[dict[str, Any]] = []
    predictions: dict[str, pd.Series] = {}

    def record_candidate(
        candidate_id: str,
        model_family: str,
        model_name: str,
        predicted_model_scale,
        *,
        order=None,
        seasonal_order=None,
        aic=np.nan,
        bic=np.nan,
        fit_warning=None,
    ):
        predicted_model_scale = pd.Series(
            np.asarray(predicted_model_scale, dtype=float),
            index=test_original.index,
        )
        predicted_original = pd.Series(
            np.asarray(from_model_scale(predicted_model_scale, transform), dtype=float),
            index=test_original.index,
            name='prediction',
        )
        metrics = forecast_metrics(test_original, predicted_original)
        rows.append({
            'candidate_id': candidate_id,
            'model_family': model_family,
            'model_name': model_name,
            'order': order,
            'seasonal_order': seasonal_order,
            'AIC': aic,
            'BIC': bic,
            **metrics,
            'status': 'success',
            'fit_warning': fit_warning,
        })
        predictions[candidate_id] = predicted_original

    # Baselines
    record_candidate(
        'naive', 'Naive', 'Naïve',
        np.repeat(training.iloc[-1], horizon),
    )
    if len(training) >= 2 * period:
        record_candidate(
            'seasonal_naive', 'Seasonal naive', f'Seasonal naïve (m={period})',
            seasonal_naive_forecast(training, horizon, period),
        )
    record_candidate(
        'drift', 'Drift', 'Drift',
        drift_forecast(training, horizon),
    )

    # Exponential smoothing
    try:
        holt_fit = ExponentialSmoothing(
            training,
            trend='add',
            damped_trend=True,
            initialization_method='estimated',
        ).fit(optimized=True, use_brute=True)
        record_candidate(
            'holt_damped', 'Holt', 'Holt damped trend',
            holt_fit.forecast(horizon),
            aic=getattr(holt_fit, 'aic', np.nan),
            bic=getattr(holt_fit, 'bic', np.nan),
        )
    except Exception as error:
        rows.append({
            'candidate_id': 'holt_damped', 'model_family': 'Holt',
            'model_name': 'Holt damped trend', 'order': None, 'seasonal_order': None,
            'AIC': np.nan, 'BIC': np.nan, 'MAE': np.nan, 'RMSE': np.nan,
            'sMAPE_pct': np.nan, 'status': f'failed: {type(error).__name__}',
        })

    allow_seasonality = len(training) >= 3 * period
    if allow_seasonality:
        try:
            ets_fit = ExponentialSmoothing(
                training,
                trend='add',
                damped_trend=True,
                seasonal='add',
                seasonal_periods=period,
                initialization_method='estimated',
            ).fit(optimized=True, use_brute=True)
            record_candidate(
                'ets_seasonal', 'ETS seasonal', f'Seasonal ETS (m={period})',
                ets_fit.forecast(horizon),
                aic=getattr(ets_fit, 'aic', np.nan),
                bic=getattr(ets_fit, 'bic', np.nan),
            )
        except Exception as error:
            rows.append({
                'candidate_id': 'ets_seasonal', 'model_family': 'ETS seasonal',
                'model_name': f'Seasonal ETS (m={period})', 'order': None,
                'seasonal_order': None, 'AIC': np.nan, 'BIC': np.nan,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE_pct': np.nan,
                'status': f'failed: {type(error).__name__}',
            })

    # SARIMA candidates
    for order, seasonal_order in build_sarima_candidates(
        differencing_order,
        period,
        allow_seasonality=config['frequency'] == 'MS' and allow_seasonality,
    ):
        candidate_id = f'sarima_{order}_{seasonal_order}'
        model_name = f'SARIMA{order}×{seasonal_order}'
        try:
            model = SARIMAX(
                training,
                order=order,
                seasonal_order=seasonal_order,
                trend='c' if differencing_order == 0 else 'n',
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            with warnings.catch_warnings(record=True) as caught_warnings:
                warnings.simplefilter('always')
                fit = model.fit(disp=False, maxiter=200)
            fit_warning = ' | '.join(dict.fromkeys(
                str(item.message) for item in caught_warnings
            )) or None
            predicted = fit.get_forecast(steps=horizon).predicted_mean
            record_candidate(
                candidate_id,
                'SARIMA',
                model_name,
                predicted,
                order=order,
                seasonal_order=seasonal_order,
                aic=fit.aic,
                bic=fit.bic,
                fit_warning=fit_warning,
            )
        except Exception as error:
            rows.append({
                'candidate_id': candidate_id, 'model_family': 'SARIMA',
                'model_name': model_name, 'order': order,
                'seasonal_order': seasonal_order, 'AIC': np.nan, 'BIC': np.nan,
                'MAE': np.nan, 'RMSE': np.nan, 'sMAPE_pct': np.nan,
                'status': f'failed: {type(error).__name__}',
            })

    leaderboard = pd.DataFrame(rows)
    successful = leaderboard[leaderboard['status'] == 'success'].copy()
    if successful.empty:
        raise RuntimeError(f"No model candidate succeeded for {config['label']}.")

    successful = successful.sort_values(
        ['RMSE', 'MAE', 'sMAPE_pct', 'AIC'],
        na_position='last',
    ).reset_index(drop=True)
    failed = leaderboard[leaderboard['status'] != 'success']
    leaderboard = pd.concat([successful, failed], ignore_index=True)
    return leaderboard, predictions


## 9. Run chronological holdout comparison for all eight series

The final observations are held out in time order. All transformations, ADF tests and model fits use only the preceding training observations. This prevents future observations entering model fitting, but the holdout is still used for model selection.


In [10]:
all_leaderboards = []
selected_model_rows = []
stationarity_rows = []
backtest_forecasts = {}
train_test_splits = {}

for series_code, config in DATASET_CONFIG.items():
    full_series = model_series[series_code]
    test_size = int(config['test_size'])

    if len(full_series) <= test_size + 24:
        raise ValueError(
            f'{series_code}: insufficient observations for a {test_size}-period holdout.'
        )

    training_original = full_series.iloc[:-test_size]
    test_original = full_series.iloc[-test_size:]
    train_test_splits[series_code] = (training_original, test_original)

    training_model_scale = to_model_scale(training_original, config['transform'])
    differencing_order, adf_table = choose_differencing_order(training_model_scale)
    adf_table.insert(0, 'series_code', series_code)
    stationarity_rows.append(adf_table)

    leaderboard, candidate_predictions = evaluate_model_candidates(
        training_original,
        test_original,
        config,
        differencing_order,
    )
    leaderboard.insert(0, 'series_code', series_code)
    leaderboard.insert(1, 'label', config['label'])
    all_leaderboards.append(leaderboard)

    selected = leaderboard[leaderboard['status'] == 'success'].iloc[0].to_dict()
    selected['differencing_order'] = differencing_order
    selected['training_observations'] = len(training_original)
    selected['test_observations'] = len(test_original)
    selected_model_rows.append(selected)

    best_prediction = candidate_predictions[selected['candidate_id']]
    backtest_forecasts[series_code] = pd.DataFrame({
        'actual': test_original,
        'predicted': best_prediction,
    })

    print(
        f"{series_code}: selected {selected['model_name']} | "
        f"RMSE={selected['RMSE']:.4f} | sMAPE={selected['sMAPE_pct']:.2f}%"
    )

model_leaderboard = pd.concat(all_leaderboards, ignore_index=True)
selected_models = pd.DataFrame(selected_model_rows).reset_index(drop=True)
stationarity_report = pd.concat(stationarity_rows, ignore_index=True)

display(selected_models[[
    'series_code', 'label', 'model_name', 'RMSE', 'MAE', 'sMAPE_pct',
    'differencing_order', 'training_observations', 'test_observations'
]])


LPMAUYN: selected SARIMA(1, 1, 1)×(0, 1, 1, 12) | RMSE=18135.0708 | sMAPE=0.45%


IUDBEDR: selected SARIMA(0, 1, 0)×(0, 0, 0, 0) | RMSE=0.0000 | sMAPE=0.00%


IUDSOIA: selected SARIMA(0, 1, 1)×(0, 0, 0, 0) | RMSE=0.0009 | sMAPE=0.02%


IUMBV34: selected SARIMA(1, 1, 0)×(1, 0, 0, 12) | RMSE=0.3909 | sMAPE=7.69%


IUMBV42: selected Holt damped trend | RMSE=0.3295 | sMAPE=6.38%


LPMVZRI: selected Seasonal ETS (m=12) | RMSE=1689.2771 | sMAPE=0.28%


XUDLUSS: selected SARIMA(1, 1, 1)×(0, 0, 0, 0) | RMSE=0.0091 | sMAPE=0.60%


XUDLERS: selected SARIMA(0, 0, 2)×(0, 0, 0, 0) | RMSE=0.0076 | sMAPE=0.59%


,series_code,label,model_name,RMSE,MAE,sMAPE_pct,differencing_order,training_observations,test_observations
0,LPMAUYN,M4 / monetary aggregate,"SARIMA(1, 1, 1)×(0, 1, 1, 12)","18,135.0708","14,505.2488",0.4497,1,53,12
1,IUDBEDR,Official Bank Rate,"SARIMA(0, 1, 0)×(0, 0, 0, 0)",0.0000,0.0000,0.0000,1,1419,30
2,IUDSOIA,SONIA,"SARIMA(0, 1, 1)×(0, 0, 0, 0)",0.0009,0.0008,0.0213,1,1418,30
3,IUMBV34,2-year fixed mortgage rate,"SARIMA(1, 1, 0)×(1, 0, 0, 12)",0.3909,0.3380,7.6864,1,54,12
4,IUMBV42,5-year fixed mortgage rate,Holt damped trend,0.3295,0.2773,6.3781,1,54,12
5,LPMVZRI,Consumer credit,Seasonal ETS (m=12),"1,689.2771","1,468.6196",0.2847,1,39,12
6,XUDLUSS,GBP/USD,"SARIMA(1, 1, 1)×(0, 0, 0, 0)",0.0091,0.0081,0.6040,1,1419,30
7,XUDLERS,GBP/EUR,"SARIMA(0, 0, 2)×(0, 0, 0, 0)",0.0076,0.0068,0.5858,0,1419,30


In [11]:
def plot_backtest(series_code: str) -> go.Figure:
    config = DATASET_CONFIG[series_code]
    training, test = train_test_splits[series_code]
    comparison = backtest_forecasts[series_code]
    selected = selected_models.loc[selected_models['series_code'] == series_code].iloc[0]

    history_window = max(len(test) * 4, 50)
    recent_training = training.iloc[-history_window:]

    figure = go.Figure()
    figure.add_trace(go.Scatter(
        x=recent_training.index,
        y=recent_training.values,
        mode='lines',
        name='Training history',
    ))
    figure.add_trace(go.Scatter(
        x=test.index,
        y=test.values,
        mode='lines+markers',
        name='Actual holdout',
    ))
    figure.add_trace(go.Scatter(
        x=comparison.index,
        y=comparison['predicted'],
        mode='lines+markers',
        name='Selected-model prediction',
    ))
    figure.update_layout(
        title=(
            f"{config['label']} — holdout evaluation<br>"
            f"<sup>{selected['model_name']} | RMSE {selected['RMSE']:.4f} | "
            f"sMAPE {selected['sMAPE_pct']:.2f}%</sup>"
        ),
        xaxis_title='Date',
        yaxis_title=config['value_column'],
        hovermode='x unified',
        template='plotly_white',
    )
    return figure


for series_code in DATASET_CONFIG:
    plot_backtest(series_code).show()

plot_backtest('IUMBV34').write_image(
    IMAGES_DIR / 'mortgage_backtest.png', width=1200, height=700, scale=2
)


## 10. Refit selected models and generate final forecasts

Selected models are refitted on all available observations. SARIMA uses model-derived 95% prediction intervals. Baselines, Holt and ETS use approximate residual-based bands; these are heuristic and should not be presented as fully calibrated uncertainty estimates.


In [12]:
def approximate_intervals(
    predicted_model_scale: np.ndarray,
    residuals: pd.Series,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray]:
    residual_std = float(np.nanstd(np.asarray(residuals, dtype=float), ddof=1))
    if not np.isfinite(residual_std) or residual_std == 0:
        residual_std = 1e-9
    widening = np.sqrt(np.arange(1, horizon + 1))
    margin = 1.96 * residual_std * widening
    return predicted_model_scale - margin, predicted_model_scale + margin


def fit_selected_model_on_full_series(
    full_original: pd.Series,
    config: dict[str, Any],
    selected: pd.Series,
) -> tuple[pd.DataFrame, pd.Series, Any]:
    transform = config['transform']
    series = to_model_scale(full_original, transform)
    horizon = int(config['forecast_horizon'])
    period = int(config['seasonal_period'])
    future_index = pd.date_range(
        start=full_original.index[-1],
        periods=horizon + 1,
        freq=config['frequency'],
    )[1:]

    family = selected['model_family']
    model_object = None
    interval_method = 'approximate residual interval'
    fit_warning = None

    if family == 'Naive':
        predicted = np.repeat(series.iloc[-1], horizon)
        residuals = series.diff().dropna()
        lower, upper = approximate_intervals(predicted, residuals, horizon)

    elif family == 'Seasonal naive':
        predicted = seasonal_naive_forecast(series, horizon, period)
        residuals = (series - series.shift(period)).dropna()
        lower, upper = approximate_intervals(predicted, residuals, horizon)

    elif family == 'Drift':
        predicted = drift_forecast(series, horizon)
        slope = (series.iloc[-1] - series.iloc[0]) / max(len(series) - 1, 1)
        residuals = (series.diff() - slope).dropna()
        lower, upper = approximate_intervals(predicted, residuals, horizon)

    elif family == 'Holt':
        model_object = ExponentialSmoothing(
            series,
            trend='add',
            damped_trend=True,
            initialization_method='estimated',
        ).fit(optimized=True, use_brute=True)
        predicted = np.asarray(model_object.forecast(horizon), dtype=float)
        residuals = pd.Series(model_object.resid, index=series.index).dropna()
        lower, upper = approximate_intervals(predicted, residuals, horizon)

    elif family == 'ETS seasonal':
        model_object = ExponentialSmoothing(
            series,
            trend='add',
            damped_trend=True,
            seasonal='add',
            seasonal_periods=period,
            initialization_method='estimated',
        ).fit(optimized=True, use_brute=True)
        predicted = np.asarray(model_object.forecast(horizon), dtype=float)
        residuals = pd.Series(model_object.resid, index=series.index).dropna()
        lower, upper = approximate_intervals(predicted, residuals, horizon)

    elif family == 'SARIMA':
        order = tuple(selected['order'])
        seasonal_order = tuple(selected['seasonal_order'])
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter('always')
            model_object = SARIMAX(
                series,
                order=order,
                seasonal_order=seasonal_order,
                trend='c' if int(selected['differencing_order']) == 0 else 'n',
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False, maxiter=300)
        fit_warning = ' | '.join(dict.fromkeys(
            str(item.message) for item in caught_warnings
        )) or None
        forecast_result = model_object.get_forecast(steps=horizon)
        predicted = np.asarray(forecast_result.predicted_mean, dtype=float)
        confidence = forecast_result.conf_int(alpha=0.05)
        lower = np.asarray(confidence.iloc[:, 0], dtype=float)
        upper = np.asarray(confidence.iloc[:, 1], dtype=float)
        residuals = pd.Series(model_object.resid, index=series.index).dropna()
        interval_method = 'SARIMA model interval'

    else:
        raise ValueError(f'Unsupported selected model family: {family}')

    predicted_original = np.asarray(from_model_scale(predicted, transform), dtype=float)
    lower_original = np.asarray(from_model_scale(lower, transform), dtype=float)
    upper_original = np.asarray(from_model_scale(upper, transform), dtype=float)

    forecast_frame = pd.DataFrame({
        'Date': future_index,
        'forecast': predicted_original,
        'lower_95': lower_original,
        'upper_95': upper_original,
        'interval_method': interval_method,
        'fit_warning': fit_warning,
    })
    return forecast_frame, residuals, model_object


final_forecast_frames = []
residuals_by_series = {}
fitted_models = {}

for series_code, config in DATASET_CONFIG.items():
    selected = selected_models.loc[selected_models['series_code'] == series_code].iloc[0]
    forecast_frame, residuals, fitted_model = fit_selected_model_on_full_series(
        model_series[series_code],
        config,
        selected,
    )
    forecast_frame.insert(0, 'series_code', series_code)
    forecast_frame.insert(1, 'label', config['label'])
    forecast_frame['model_name'] = selected['model_name']
    forecast_frame['training_end'] = model_series[series_code].index.max()
    final_forecast_frames.append(forecast_frame)
    residuals_by_series[series_code] = residuals
    fitted_models[series_code] = fitted_model

final_forecasts = pd.concat(final_forecast_frames, ignore_index=True)
display(final_forecasts.groupby(['series_code', 'label', 'model_name']).head(2))


,series_code,label,Date,forecast,lower_95,upper_95,interval_method,fit_warning,model_name,training_end
0,LPMAUYN,M4 / monetary aggregate,2026-06-01,"3,285,865.3744","3,247,459.2014","3,324,725.7592",SARIMA model interval,None,"SARIMA(1, 1, 1)×(0, 1, 1, 12)",2026-05-01
1,LPMAUYN,M4 / monetary aggregate,2026-07-01,"3,287,601.7818","3,233,726.3503","3,342,374.8038",SARIMA model interval,None,"SARIMA(1, 1, 1)×(0, 1, 1, 12)",2026-05-01
12,IUDBEDR,Official Bank Rate,2026-07-24,3.7500,3.6662,3.8338,SARIMA model interval,None,"SARIMA(0, 1, 0)×(0, 0, 0, 0)",2026-07-23
13,IUDBEDR,Official Bank Rate,2026-07-27,3.7500,3.6315,3.8685,SARIMA model interval,None,"SARIMA(0, 1, 0)×(0, 0, 0, 0)",2026-07-23
42,IUDSOIA,SONIA,2026-07-23,3.7303,3.6469,3.8137,SARIMA model interval,Maximum Likelihood optimization failed to conv...,"SARIMA(0, 1, 1)×(0, 0, 0, 0)",2026-07-22
43,IUDSOIA,SONIA,2026-07-24,3.7303,3.6122,3.8484,SARIMA model interval,Maximum Likelihood optimization failed to conv...,"SARIMA(0, 1, 1)×(0, 0, 0, 0)",2026-07-22
72,IUMBV34,2-year fixed mortgage rate,2026-07-01,4.7649,4.0823,5.4476,SARIMA model interval,None,"SARIMA(1, 1, 0)×(1, 0, 0, 12)",2026-06-01
73,IUMBV34,2-year fixed mortgage rate,2026-08-01,4.7466,3.5644,5.9289,SARIMA model interval,None,"SARIMA(1, 1, 0)×(1, 0, 0, 12)",2026-06-01
84,IUMBV42,5-year fixed mortgage rate,2026-07-01,4.6367,4.0500,5.2234,approximate residual interval,None,Holt damped trend,2026-06-01
85,IUMBV42,5-year fixed mortgage rate,2026-08-01,4.6261,3.7963,5.4558,approximate residual interval,None,Holt damped trend,2026-06-01


## 11. Residual diagnostics

Ljung–Box tests check for evidence of remaining autocorrelation at the selected lag; failure to reject does not prove independence. Jarque–Bera tests assess normality. Diagnostic results are model checks rather than guarantees of forecast validity.


In [13]:
diagnostic_rows = []

for series_code, residuals in residuals_by_series.items():
    clean_residuals = pd.Series(residuals).replace([np.inf, -np.inf], np.nan).dropna()
    burn_in = min(max(5, DATASET_CONFIG[series_code]['seasonal_period']), max(len(clean_residuals) // 10, 0))
    if burn_in > 0:
        clean_residuals = clean_residuals.iloc[burn_in:]

    if len(clean_residuals) < 10:
        diagnostic_rows.append({
            'series_code': series_code,
            'label': DATASET_CONFIG[series_code]['label'],
            'residual_observations': len(clean_residuals),
            'ljung_box_lag': np.nan,
            'ljung_box_p_value': np.nan,
            'jarque_bera_statistic': np.nan,
            'jarque_bera_p_value': np.nan,
            'residual_autocorrelation_assessment': 'insufficient residuals',
        })
        continue

    lag = min(10, max(1, len(clean_residuals) // 5))
    ljung_box = acorr_ljungbox(clean_residuals, lags=[lag], return_df=True)
    jb_result = jarque_bera(clean_residuals)
    lb_p_value = float(ljung_box['lb_pvalue'].iloc[-1])

    diagnostic_rows.append({
        'series_code': series_code,
        'label': DATASET_CONFIG[series_code]['label'],
        'residual_observations': len(clean_residuals),
        'ljung_box_lag': lag,
        'ljung_box_p_value': lb_p_value,
        'jarque_bera_statistic': float(jb_result.statistic),
        'jarque_bera_p_value': float(jb_result.pvalue),
        'residual_autocorrelation_assessment': (
            'no significant autocorrelation detected' if lb_p_value >= 0.05
            else 'remaining autocorrelation detected'
        ),
    })

residual_diagnostics = pd.DataFrame(diagnostic_rows)
display(residual_diagnostics)


,series_code,label,residual_observations,ljung_box_lag,ljung_box_p_value,jarque_bera_statistic,jarque_bera_p_value,residual_autocorrelation_assessment
0,LPMAUYN,M4 / monetary aggregate,59,10,0.9973,"6,962.4704",0.0000,no significant autocorrelation detected
1,IUDBEDR,Official Bank Rate,1444,10,1.0000,"1,131,841.2339",0.0000,no significant autocorrelation detected
2,IUDSOIA,SONIA,1443,10,1.0000,"1,136,591.7011",0.0000,no significant autocorrelation detected
3,IUMBV34,2-year fixed mortgage rate,60,10,0.1940,164.2937,0.0000,no significant autocorrelation detected
4,IUMBV42,5-year fixed mortgage rate,60,10,0.0539,114.5657,0.0000,no significant autocorrelation detected
5,LPMVZRI,Consumer credit,46,9,0.5216,2.0696,0.3553,no significant autocorrelation detected
6,XUDLUSS,GBP/USD,1444,10,0.2303,466.8387,0.0000,no significant autocorrelation detected
7,XUDLERS,GBP/EUR,1444,10,0.0000,1.8363,0.3992,remaining autocorrelation detected


In [14]:
def plot_residuals(series_code: str) -> go.Figure:
    residuals = pd.Series(residuals_by_series[series_code]).dropna()
    selected = selected_models.loc[selected_models['series_code'] == series_code].iloc[0]
    figure = px.histogram(
        x=residuals,
        nbins=40,
        marginal='box',
        labels={'x': 'Residual'},
        title=f"{DATASET_CONFIG[series_code]['label']} residual distribution — {selected['model_name']}",
        template='plotly_white',
    )
    return figure


for series_code in DATASET_CONFIG:
    plot_residuals(series_code).show()


## 12. Final forecast visualisations


In [15]:
def plot_final_forecast(series_code: str, history_points: int | None = None) -> go.Figure:
    config = DATASET_CONFIG[series_code]
    history = model_series[series_code]
    forecast = final_forecasts[final_forecasts['series_code'] == series_code].copy()
    selected = selected_models.loc[selected_models['series_code'] == series_code].iloc[0]

    if history_points is None:
        history_points = 48 if config['frequency'] == 'MS' else 180
    history = history.iloc[-history_points:]

    figure = go.Figure()
    figure.add_trace(go.Scatter(
        x=history.index,
        y=history.values,
        mode='lines',
        name='Historical',
    ))
    figure.add_trace(go.Scatter(
        x=forecast['Date'],
        y=forecast['upper_95'],
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    ))
    figure.add_trace(go.Scatter(
        x=forecast['Date'],
        y=forecast['lower_95'],
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        name='95% interval',
        hoverinfo='skip',
    ))
    figure.add_trace(go.Scatter(
        x=forecast['Date'],
        y=forecast['forecast'],
        mode='lines+markers',
        name='Forecast',
    ))
    figure.update_layout(
        title=(
            f"{config['label']} forecast<br>"
            f"<sup>{selected['model_name']} | holdout RMSE {selected['RMSE']:.4f}</sup>"
        ),
        xaxis_title='Date',
        yaxis_title=config['value_column'],
        hovermode='x unified',
        template='plotly_white',
    )
    return figure


for series_code in DATASET_CONFIG:
    plot_final_forecast(series_code).show()

plot_final_forecast('LPMAUYN').write_image(
    IMAGES_DIR / 'm4_forecast.png', width=1200, height=700, scale=2
)
plot_final_forecast('XUDLUSS').write_image(
    IMAGES_DIR / 'gbp_usd_forecast.png', width=1200, height=700, scale=2
)


## 13. Deterministic natural-language query layer

This local convenience interface uses transparent keyword matching and regular expressions. It does not call an LLM, perform semantic search or generate new economic reasoning. It identifies a requested series and horizon, then formats the corresponding stored statistical forecast.


In [16]:
SERIES_KEYWORDS = {
    'LPMAUYN': ['m4', 'money supply', 'monetary aggregate', 'broad money'],
    'IUDBEDR': ['bank rate', 'base rate', 'official rate', 'interest rate'],
    'IUDSOIA': ['sonia', 'overnight rate', 'sterling overnight'],
    'IUMBV34': ['2 year mortgage', '2-year mortgage', 'two year mortgage', '2y mortgage'],
    'IUMBV42': ['5 year mortgage', '5-year mortgage', 'five year mortgage', '5y mortgage'],
    'LPMVZRI': ['consumer credit', 'credit amount', 'household credit'],
    'XUDLUSS': ['gbp usd', 'gbp/usd', 'pound dollar', 'sterling dollar', 'dollar exchange'],
    'XUDLERS': ['gbp eur', 'gbp/eur', 'pound euro', 'sterling euro', 'euro exchange'],
}


def normalise_question(question: str) -> str:
    return re.sub(r'[^a-z0-9/\- ]+', ' ', question.lower()).strip()


def resolve_series_code(question: str) -> str:
    normalised = normalise_question(question)
    scores = {}
    for series_code, keywords in SERIES_KEYWORDS.items():
        scores[series_code] = sum(len(keyword) for keyword in keywords if keyword in normalised)

    best_code = max(scores, key=scores.get)
    if scores[best_code] == 0:
        available = ', '.join(config['label'] for config in DATASET_CONFIG.values())
        raise ValueError(f'Could not identify a series. Try one of: {available}')
    return best_code


def extract_requested_steps(question: str, series_code: str) -> int | None:
    normalised = normalise_question(question)
    match = re.search(r'\b(\d{1,3})\s*(day|days|business day|business days|month|months|period|periods)\b', normalised)
    if not match:
        return None

    number = int(match.group(1))
    unit = match.group(2)
    frequency = DATASET_CONFIG[series_code]['frequency']

    if frequency == 'MS' and 'day' in unit:
        number = max(1, round(number / 30))
    elif frequency == 'B' and 'month' in unit:
        number = number * 21

    maximum = int(DATASET_CONFIG[series_code]['forecast_horizon'])
    return min(max(number, 1), maximum)


def forecast_summary(series_code: str, steps: int | None = None) -> str:
    config = DATASET_CONFIG[series_code]
    history = model_series[series_code]
    forecast = final_forecasts[final_forecasts['series_code'] == series_code].copy()
    selected = selected_models.loc[selected_models['series_code'] == series_code].iloc[0]

    if steps is None:
        steps = len(forecast)
    steps = min(max(int(steps), 1), len(forecast))
    target = forecast.iloc[steps - 1]

    latest_value = float(history.iloc[-1])
    target_value = float(target['forecast'])
    absolute_change = target_value - latest_value
    percentage_change = (absolute_change / abs(latest_value) * 100) if latest_value != 0 else np.nan

    tolerance = max(abs(latest_value) * 0.001, 1e-9)
    if absolute_change > tolerance:
        direction = 'increase'
    elif absolute_change < -tolerance:
        direction = 'decrease'
    else:
        direction = 'remain broadly stable'

    unit_name = 'month' if config['frequency'] == 'MS' else 'business day'
    plural = '' if steps == 1 else 's'

    return (
        f"EconLens identifies {config['label']}. The latest observed value is {latest_value:,.4f}. "
        f"The selected {selected['model_name']} model forecasts {target_value:,.4f} after "
        f"{steps} {unit_name}{plural}, implying the series may {direction}. "
        f"The estimated change is {absolute_change:,.4f} ({percentage_change:,.2f}%). "
        f"The 95% forecast interval at that horizon is {target['lower_95']:,.4f} to "
        f"{target['upper_95']:,.4f}. Holdout RMSE was {selected['RMSE']:,.4f} and "
        f"sMAPE was {selected['sMAPE_pct']:,.2f}%. This is a statistical forecast, not a causal claim."
    )


def ask_econlens(question: str, show_chart: bool = True) -> str:
    series_code = resolve_series_code(question)
    steps = extract_requested_steps(question, series_code)
    answer = forecast_summary(series_code, steps)
    print(answer)
    if show_chart:
        plot_final_forecast(series_code).show()
    return answer

plot_final_forecast('LPMAUYN').write_image(
    IMAGES_DIR / 'm4_forecast.png', width=1200, height=700, scale=2
)
plot_final_forecast('XUDLUSS').write_image(
    IMAGES_DIR / 'gbp_usd_forecast.png', width=1200, height=700, scale=2
)


In [17]:
# Example questions — replace these with your own.
example_questions = [
    'What is the 12 month forecast for M4 money supply?',
    'Where could the Bank Rate be in 20 business days?',
    'Forecast the 2-year mortgage rate for 6 months.',
    'What does EconLens expect for GBP/USD over the next month?',
]

for question in example_questions:
    print(f'\nQUESTION: {question}')
    ask_econlens(question, show_chart=False)



QUESTION: What is the 12 month forecast for M4 money supply?
EconLens identifies M4 / monetary aggregate. The latest observed value is 3,278,498.0000. The selected SARIMA(1, 1, 1)×(0, 1, 1, 12) model forecasts 3,365,192.5570 after 12 months, implying the series may increase. The estimated change is 86,694.5570 (2.64%). The 95% forecast interval at that horizon is 3,233,205.1033 to 3,502,568.0661. Holdout RMSE was 18,135.0708 and sMAPE was 0.45%. This is a statistical forecast, not a causal claim.

QUESTION: Where could the Bank Rate be in 20 business days?
EconLens identifies Official Bank Rate. The latest observed value is 3.7500. The selected SARIMA(0, 1, 0)×(0, 0, 0, 0) model forecasts 3.7500 after 20 business days, implying the series may remain broadly stable. The estimated change is 0.0000 (0.00%). The 95% forecast interval at that horizon is 3.3751 to 4.1249. Holdout RMSE was 0.0000 and sMAPE was 0.00%. This is a statistical forecast, not a causal claim.

QUESTION: Forecast the

## 14. Save portfolio-ready outputs


In [18]:
leaderboard_path = OUTPUT_DIR / 'econlens_model_leaderboard.csv'
selected_models_path = OUTPUT_DIR / 'econlens_selected_models.csv'
stationarity_path = OUTPUT_DIR / 'econlens_stationarity_report.csv'
diagnostics_path = OUTPUT_DIR / 'econlens_residual_diagnostics.csv'
forecasts_path = OUTPUT_DIR / 'econlens_final_forecasts.csv'
backtests_path = OUTPUT_DIR / 'econlens_backtest_predictions.csv'

backtest_output = pd.concat([
    frame.reset_index(names='Date').assign(
        series_code=series_code,
        label=DATASET_CONFIG[series_code]['label'],
    )
    for series_code, frame in backtest_forecasts.items()
], ignore_index=True)

# Convert tuple-valued model parameters to strings for clean CSV exports.
leaderboard_export = model_leaderboard.copy()
selected_models_export = selected_models.copy()
for export_frame in [leaderboard_export, selected_models_export]:
    export_frame['order'] = export_frame['order'].astype(str)
    export_frame['seasonal_order'] = export_frame['seasonal_order'].astype(str)

leaderboard_export.to_csv(leaderboard_path, index=False)
selected_models_export.to_csv(selected_models_path, index=False)
stationarity_report.to_csv(stationarity_path, index=False)
residual_diagnostics.to_csv(diagnostics_path, index=False)
final_forecasts.to_csv(forecasts_path, index=False)
backtest_output.to_csv(backtests_path, index=False)

print('Saved outputs:')
for path in [
    leaderboard_path,
    selected_models_path,
    stationarity_path,
    diagnostics_path,
    forecasts_path,
    backtests_path,
]:
    print(f'- {path}')


Saved outputs:
- C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_model_leaderboard.csv
- C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_selected_models.csv
- C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_stationarity_report.csv
- C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_residual_diagnostics.csv
- C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_final_forecasts.csv
- C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_backtest_predictions.csv


## 15. Final model card

This table summarizes the selected candidate, model-selection holdout error, forecast direction and residual diagnostic for each series. A zero Bank Rate holdout error reflects an unchanged 30-business-day holdout and must not be interpreted as proof that policy decisions are predictable.


In [19]:
model_card_rows = []

for series_code, config in DATASET_CONFIG.items():
    selected = selected_models.loc[selected_models['series_code'] == series_code].iloc[0]
    diagnostics = residual_diagnostics.loc[
        residual_diagnostics['series_code'] == series_code
    ].iloc[0]
    forecast = final_forecasts[final_forecasts['series_code'] == series_code]
    latest = float(model_series[series_code].iloc[-1])
    final_value = float(forecast.iloc[-1]['forecast'])

    model_card_rows.append({
        'series_code': series_code,
        'label': config['label'],
        'frequency': config['frequency_label'],
        'selected_model': selected['model_name'],
        'holdout_RMSE': selected['RMSE'],
        'holdout_sMAPE_pct': selected['sMAPE_pct'],
        'latest_observation': latest,
        'forecast_horizon': config['forecast_horizon'],
        'forecast_endpoint': final_value,
        'forecast_change_pct': ((final_value / latest) - 1) * 100 if latest != 0 else np.nan,
        'residual_assessment': diagnostics['residual_autocorrelation_assessment'],
    })

model_card = pd.DataFrame(model_card_rows)
model_card_path = OUTPUT_DIR / 'econlens_model_card.csv'
model_card.to_csv(model_card_path, index=False)
display(model_card)
print(f'Saved model card: {model_card_path}')

model_card_image = model_card[[
    'label', 'selected_model', 'holdout_RMSE', 'holdout_sMAPE_pct'
]].copy()
model_card_image['holdout_RMSE'] = model_card_image['holdout_RMSE'].map(lambda x: f'{x:,.4f}')
model_card_image['holdout_sMAPE_pct'] = model_card_image['holdout_sMAPE_pct'].map(lambda x: f'{x:.2f}%')
model_card_figure = go.Figure(data=[go.Table(
    header=dict(values=['Series', 'Selected model', 'RMSE', 'sMAPE'], fill_color='#17324d', font_color='white', align='left'),
    cells=dict(values=[model_card_image[column] for column in model_card_image.columns], fill_color='#f3f6f9', align='left'),
)])
model_card_figure.update_layout(title='Selected models and chronological holdout results', margin=dict(l=20, r=20, t=60, b=20))
model_card_figure.write_image(IMAGES_DIR / 'model_comparison.png', width=1400, height=650, scale=2)


,series_code,label,frequency,selected_model,holdout_RMSE,holdout_sMAPE_pct,latest_observation,forecast_horizon,forecast_endpoint,forecast_change_pct,residual_assessment
0,LPMAUYN,M4 / monetary aggregate,monthly,"SARIMA(1, 1, 1)×(0, 1, 1, 12)","18,135.0708",0.4497,"3,278,498.0000",12,"3,365,192.5570",2.6443,no significant autocorrelation detected
1,IUDBEDR,Official Bank Rate,business daily,"SARIMA(0, 1, 0)×(0, 0, 0, 0)",0.0000,0.0000,3.7500,30,3.7500,0.0000,no significant autocorrelation detected
2,IUDSOIA,SONIA,business daily,"SARIMA(0, 1, 1)×(0, 0, 0, 0)",0.0009,0.0213,3.7303,30,3.7303,0.0000,no significant autocorrelation detected
3,IUMBV34,2-year fixed mortgage rate,monthly,"SARIMA(1, 1, 0)×(1, 0, 0, 12)",0.3909,7.6864,4.8100,12,4.7302,-1.6587,no significant autocorrelation detected
4,IUMBV42,5-year fixed mortgage rate,monthly,Holt damped trend,0.3295,6.3781,4.6500,12,4.5881,-1.3321,no significant autocorrelation detected
5,LPMVZRI,Consumer credit,monthly,Seasonal ETS (m=12),"1,689.2771",0.2847,"530,094.0000",12,"573,133.7046",8.1193,no significant autocorrelation detected
6,XUDLUSS,GBP/USD,business daily,"SARIMA(1, 1, 1)×(0, 0, 0, 0)",0.0091,0.6040,1.3320,30,1.3331,0.0862,no significant autocorrelation detected
7,XUDLERS,GBP/EUR,business daily,"SARIMA(0, 0, 2)×(0, 0, 0, 0)",0.0076,0.5858,1.1713,30,1.1660,-0.4525,remaining autocorrelation detected


Saved model card: C:\Users\Windows\.codex\.chatgpt-projects\g-p-69fcedc398848191b84b3e55febd9f71\econlens-uk-forecasting\outputs\econlens_model_card.csv


## Project completion criteria

Before publication:

- place the eight documented Bank of England CSVs in `data/raw/`;
- restart the kernel and run every cell from the repository root;
- inspect all convergence or numerical warnings rather than suppressing them;
- confirm that every candidate status and exported output is reproducible;
- retain representative charts and compact result tables;
- describe reported metrics as single-holdout model-comparison results;
- describe non-SARIMA uncertainty bands as approximate;
- disclose that the query layer is deterministic keyword routing; and
- keep the limitations visible in the README.
